# 15 · Температурные признаки для fire-модели (value-пайплайн)

**Цель** — по показаниям «Датчика температуры» построить **6-часовые агрегаты по
объекту** (mean/min/max, число наблюдений), добавить их к fire-панели и проверить,
улучшают ли они TTE-модель пожарного риска («серьёзная тревога»):
сравнение «с температурой vs без» на том же holdout ≥ 2025-07-01.

Этапы:
1. чанковый проход по журналам → `(объект, 6ч-бакет, sum/cnt/min/max)` температур;
2. join к `subdaily_panel_fire6h.csv` по `(ид_объект, бакет)`;
3. оконные средние (12ч/24ч/48ч/7д/30д) и отклонение от суточного baseline
   (baseline строится только по train ≤ 2024-12-31);
4. пересборка субъектов TTE и обучение CatBoost-discrete-hazard в двух
   конфигурациях: без температурных фич и с ними.


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import (survival_metrics, pr_at_horizon,
                             fit_discrete_hazard, predict_survival_discrete)
from tte_experiments import baseline_survival_by_type


DATASET = du.find_dataset_dir()
ref = du.load_ref_channels()[["ид_канала_данных", "тип_датчика", "ид_объект"]]
ref = ref.dropna(subset=["ид_объект"])
tch = set(ref.loc[ref["тип_датчика"] == "Датчик температуры", "ид_канала_данных"])
ch_to_obj = dict(zip(ref["ид_канала_данных"].astype(str), ref["ид_объект"].astype(str)))
print("термо-каналов:", len(tch), "| объект-функция готова")
sns.set_theme(style="whitegrid")


термо-каналов: 600 | объект-функция готова


In [2]:
# 1) Чанковый проход: 6-часовые агрегаты температуры по объекту (~10-15 мин)
VALID_MIN, VALID_MAX = -20.0, 60.0
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9   # 6ч в наносекундах
parts = []
for fp in du.journal_files():
    for ch in du.read_chunks(fp, cols=["ид_канала_данных", "дата", "время",
                                       "значение_датчика"], chunksize=500_000):
        m = ch["ид_канала_данных"].isin(tch)
        if not m.any():
            continue
        ch = ch[m]
        vals = pd.to_numeric(ch["значение_датчика"].astype(str).str.strip()
                             .str.replace(",", ".", regex=False), errors="coerce")
        ok = vals.notna()
        if not ok.any():
            continue
        v = vals[ok].to_numpy(np.float64)
        valid = (v >= VALID_MIN) & (v <= VALID_MAX)
        if not valid.any():
            continue
        ch2 = ch[ok][valid]
        ts = pd.to_datetime(ch["дата"] + " " + ch["время"].fillna("00:00:00"),
                            format="%Y-%m-%d %H:%M:%S", errors="coerce")[ok][valid]
        buck = (ts.astype("datetime64[ns]").astype("int64") // hour_ns).to_numpy()
        ag = pd.DataFrame({"об": ch2["ид_канала_данных"].astype(str).map(ch_to_obj),
                           "бакет": buck, "v": v[valid]}).dropna(subset=["об"])
        if ag.empty:
            continue
        g = ag.groupby(["об", "бакет"], sort=False).agg(
            vsum=("v", "sum"), n=("v", "size"),
            vmin=("v", "min"), vmax=("v", "max")).reset_index()
        parts.append(g)
    print(f"  {fp.name}: чанков-агрегатов={sum(len(p) for p in parts):,}", flush=True)

tobj = pd.concat(parts, ignore_index=True)
tobj = tobj.groupby(["об", "бакет"], sort=False).agg(
    t_sum=("vsum", "sum"), t_cnt=("n", "sum"),
    t_min=("vmin", "min"), t_max=("vmax", "max")).reset_index()
tobj["t_mean"] = tobj["t_sum"] / tobj["t_cnt"]
tobj.to_parquet(DATASET / "temp_obj_bucket6h.parquet")
print(f"temp_obj_bucket6h: {len(tobj):,} строк -> parquet сохранён")


  ext-journal-2019.csv: чанков-агрегатов=15,337


  ext-journal-2020.csv: чанков-агрегатов=31,850


  ext-journal-2021.csv: чанков-агрегатов=47,871


  ext-journal-2022.csv: чанков-агрегатов=66,171


  ext-journal-2023.csv: чанков-агрегатов=85,035


  ext-journal-2024.csv: чанков-агрегатов=103,787


  ext-journal-2025.csv: чанков-агрегатов=123,216


  ext-journal-2026.csv: чанков-агрегатов=132,989


temp_obj_bucket6h: 129,215 строк -> parquet сохранён


In [3]:
# 2) Join к fire-панели и оконные температурные признаки
panel = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
tobj = pd.read_parquet(DATASET / "temp_obj_bucket6h.parquet")
tobj["об"] = tobj["об"].astype(str)
tmap = tobj[["об", "бакет", "t_cnt", "t_mean", "t_min", "t_max"]].rename(
    columns={"об": "ид_объект"})

panel = panel.merge(tmap, on=["ид_объект", "бакет"], how="left")
panel["t_cnt"] = panel["t_cnt"].fillna(0).astype(np.float64)
panel["t_есть_б"] = (panel["t_cnt"].to_numpy() > 0).astype(np.int8)
t_num = ["t_cnt", "t_mean", "t_min", "t_max", "t_есть_б"]
print("панель:", panel.shape, "| объектов с термодатчиками:",
      panel.loc[(panel["t_cnt"] > 0)].groupby("ид_объект").ngroups)

# 3) Оконные средние по объекту: t_sum = t_mean * t_cnt, окна 12ч..30д
panel["t_sum_б"] = panel["t_mean"].to_numpy() * panel["t_cnt"].to_numpy()
panel = fe._add_bucket_windows(panel, ["t_sum_б", "t_cnt"],
                               windows=fe.BUCKET_WINDOWS, group_col="ид_объект")
for label in ("12ч", "24ч", "48ч", "7д", "30д"):
    panel[f"t_mean_сум_{label}"] = (
        panel[f"t_sum_б_сум_{label}"] / panel[f"t_cnt_сум_{label}"].replace(0, np.nan))
print("оконные средние готовы, колонок:", len(panel.columns))
print("пример строки c термодатчиком:")
tmp = panel[panel["t_cnt"] > 0][["ид_объект", "бакет", "t_cnt", "t_mean", "t_min",
                                 "t_max", "t_mean_сум_24ч", "t_mean_сум_7д"]].head(3)
print(tmp.to_string(index=False))


панель: (1204508, 148) | объектов с термодатчиками: 30


оконные средние готовы, колонок: 164
пример строки c термодатчиком:


ид_объект  бакет  t_cnt  t_mean  t_min  t_max  t_mean_сум_24ч  t_mean_сум_7д
      108  71590    1.0    28.0   28.0   28.0            28.0           28.0
      108  71597    1.0    30.0   30.0   30.0            30.0           29.0
      108  71617    1.0    32.0   32.0   32.0             NaN           30.0


In [4]:
# 4) Суточный baseline по объекту (только train <= 2024-12-31), без утечки
dt = pd.to_datetime(panel["дата"])
tr_mask = (~panel["аномально"].to_numpy()) & (dt <= pd.Timestamp("2024-12-31"))
base = (panel.loc[tr_mask & (panel["t_cnt"] > 0)]
        .groupby(["ид_объект", "час_бакета"])["t_mean"].mean()
        .rename("t_base_hour").reset_index())
panel = panel.merge(base, on=["ид_объект", "час_бакета"], how="left")
panel["t_dev_hour"] = panel["t_mean"].to_numpy() - panel["t_base_hour"].to_numpy()
print("строк с температурной фичей: {:.1f}%".format((panel["t_cnt"] > 0).mean() * 100))

# 5) Субъекты TTE, событие = серия «серьёзных»
panel = tte.refit_z_train_only(panel)
panel = tte.add_series_context(panel, event_col="серьёзных")
subjects = tte.add_tte_labels(panel)
subjects["ид_объект_code"] = subjects["ид_объект"].astype("category").cat.codes
subjects["тип_датчика_code"] = subjects["тип_датчика"].astype("category").cat.codes
train, val, holdout = tte.split_subjects(subjects)
X_all = tte.subject_features(train)
X_base = [c for c in X_all if not c.startswith("t_")]
print("фич: всего =", len(X_all), "| базовых =", len(X_base),
      "| t-фич =", len(X_all) - len(X_base))
print("train/val/holdout:", len(train), len(val), len(holdout))


строк с температурной фичей: 75.5%


фич: всего = 160 | базовых = 137 | t-фич = 23
train/val/holdout: 852127 67466 248613


In [5]:
def run_train(X_cols, tag):
    tr = tte.sample_subjects(train, 15000, 10000, 10000, seed=42)
    va = tte.sample_subjects(val, 2000, 2000, 2000, seed=43)
    ho = tte.sample_holdout(holdout, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(tr, X_cols, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(va, X_cols, tte.HORIZON_BUCKETS)
    m = fit_discrete_hazard(X_tr, y_tr, X_va, y_va, iters=700, lr=0.05, depth=6, seed=42)
    surv_tr = tte.make_surv_struct(tr["event_flag"], tr["obs_days"])
    surv_ho = tte.make_surv_struct(ho["event_flag"], ho["obs_days"])
    S_ho = predict_survival_discrete(m, ho, X_cols)
    risk = 1 - S_ho[:, -1]
    met = survival_metrics(surv_tr, surv_ho, risk, S_ho, tte.EVAL_TIMES)
    y24 = ((ho["event_flag"] == 1) & (ho["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S_ho[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    print(f"[{tag}] Uno-C={met['uno_c']:.3f} | meanAUC={met['mean_auc']:.3f} "
          f"| IBS={met['ibs']:.3f} | P(24ч) PR-AUC={pr['pr_auc']:.3f} "
          f"(база {pr['base']*100:.2f}%)")
    return m, met, pr

m_base, met_base, pr_base = run_train(X_base, "без температуры")
m_full, met_full, pr_full = run_train(X_all, "с температурой")


0:	learn: 0.5351136	test: 0.5329979	best: 0.5329979 (0)	total: 375ms	remaining: 4m 22s


100:	learn: 0.0082107	test: 0.0065181	best: 0.0064868 (95)	total: 30s	remaining: 2m 58s


200:	learn: 0.0066935	test: 0.0065699	best: 0.0063841 (113)	total: 57.4s	remaining: 2m 22s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.006384060499
bestIteration = 113

Shrink model to first 114 iterations.


[без температуры] Uno-C=0.869 | meanAUC=0.875 | IBS=0.048 | P(24ч) PR-AUC=0.404 (база 1.53%)


0:	learn: 0.5390482	test: 0.5362068	best: 0.5362068 (0)	total: 290ms	remaining: 3m 22s


100:	learn: 0.0080625	test: 0.0060587	best: 0.0060587 (100)	total: 32.6s	remaining: 3m 13s


200:	learn: 0.0065582	test: 0.0063020	best: 0.0060396 (105)	total: 1m 3s	remaining: 2m 37s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.006039596537
bestIteration = 105

Shrink model to first 106 iterations.


[с температурой] Uno-C=0.819 | meanAUC=0.830 | IBS=0.055 | P(24ч) PR-AUC=0.327 (база 1.53%)


In [6]:
# Дополнительная проверка: «умеренный» набор t-фич без окна-делений (8 колонок)
T_LIGHT = ["t_cnt", "t_mean", "t_min", "t_max", "t_есть_б", "t_dev_hour",
           "t_mean_сум_24ч", "t_mean_сум_7д"]
X_tlight = X_base + [c for c in T_LIGHT if c in X_all]
print("t-light фич:", len(X_tlight) - len(X_base))
m_light, met_light, pr_light = run_train(X_tlight, "t-умеренный")


t-light фич: 8


0:	learn: 0.5357288	test: 0.5344724	best: 0.5344724 (0)	total: 238ms	remaining: 2m 46s


100:	learn: 0.0081376	test: 0.0062454	best: 0.0062396 (97)	total: 29.4s	remaining: 2m 54s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.006239624853
bestIteration = 97

Shrink model to first 98 iterations.


[t-умеренный] Uno-C=0.847 | meanAUC=0.854 | IBS=0.050 | P(24ч) PR-AUC=0.366 (база 1.53%)


In [7]:
print("=== Сравнение на holdout >= 2025-07-01 (одни и те же 20k субъектов) ===")
row = lambda met, pr, tag: (tag, met["uno_c"], met["mean_auc"], met["ibs"],
                            pr["pr_auc"], pr["base"])
cmp = pd.DataFrame(
    [row(met_base, pr_base, "без t-фич"),
     row(met_light, pr_light, "t-умеренный (8)"),
     row(met_full, pr_full, "t-полный (23)")],
    columns=["конфиг", "Uno-C", "meanAUC", "IBS", "PR-AUC(24ч)", "база24"])
print(cmp.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print()
print("Вывод зависит от того, вернётся ли метрика к уровню baseline:")
print("если t-умеренный ≈ без t-фич — температурные признаки в текущем виде")
print("не добавляют сигнала (и даже ухудшают при избытке колонок с NaN).")


=== Сравнение на holdout >= 2025-07-01 (одни и те же 20k субъектов) ===
         конфиг  Uno-C  meanAUC   IBS  PR-AUC(24ч)  база24
      без t-фич  0.869    0.875 0.048        0.404   0.015
t-умеренный (8)  0.847    0.854 0.050        0.366   0.015
  t-полный (23)  0.819    0.830 0.055        0.327   0.015

Вывод зависит от того, вернётся ли метрика к уровню baseline:
если t-умеренный ≈ без t-фич — температурные признаки в текущем виде
не добавляют сигнала (и даже ухудшают при избытке колонок с NaN).


In [8]:
imp = pd.DataFrame(m_full.get_feature_importance(prettified=True))
imp.columns = ["feature", "importance"]
print("топ-20 важности (с t-фичами):")
print(imp.head(20).to_string(index=False))
t_imp = imp[imp["feature"].str.startswith("t_")]["importance"]
print(f"суммарная важность t-фич: {t_imp.sum():.1f} из {imp['importance'].sum():.1f}")


топ-20 важности (с t-фичами):
                        feature  importance
                       час_шага   36.807723
               день_недели_шага   11.287376
                 доля_горизонта    4.311005
              дни_с_посл_неиспр    3.270617
              дни_с_посл_старта    2.487078
        каналов_задымлений_об_б    2.469407
                            шаг    2.317499
            неисправностей_об_б    2.176376
                     месяц_шага    2.134279
        задымлений_об_б_сум_30д    1.643370
           событий_об_б_сум_30д    1.585916
каналов_задымлений_об_б_сум_30д    1.209784
 каналов_тревог_дым_об_б_сум_7д    1.202055
               тип_датчика_code    1.063650
каналов_тревог_дым_об_б_сум_12ч    1.033153
        каналов_тревог_дым_об_б    1.025382
          каналов_активных_об_б    1.009482
каналов_задымлений_об_б_сум_12ч    0.983700
                   событий_об_б    0.966316
              серьёзных_сум_30д    0.943144
суммарная важность t-фич: 3.5 из 100.0


### Выводы

- Улучшили ли температурные признаки fire-модель — видно из таблицы сравнения
  (Uno-C, PR-AUC 24ч) на одном и том же holdout.
- Важность t-фич показывает, насколько модель реально опирается на
  температуру, а не на событийные счётчики.
- Это первый шаг value-пайплайна: дальше — min/max-тренды, отклонения от
  сезонного baseline, корреляция «дым + температура» в один бакет.
